# Notebook 02: Detection Floor

Characterises the spatially varying detection floor: what trip lengths are detectable at each tower site.

## Setup

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA    = Path("../data")
FIGURES = Path("../figures")
FIGURES.mkdir(parents=True, exist_ok=True)
CENSUS  = DATA / "Cartografia_censo2024_R13/Cartografia_censo2024_R13.gpkg"

CRS_GEO = "EPSG:4674"

## Load site data

In [ ]:
bts = pd.read_parquet(DATA / "bts_sites.parquet")
comunas = gpd.read_file(CENSUS, layer="Comunal_CPV24")

print(f"Sites: {len(bts):,}")
print(f"Radius range: {bts['radius'].min():.0f} -- {bts['radius'].max():.0f} m")
print(f"Sectors range: {bts['n_sectors'].min()} -- {bts['n_sectors'].max()}")

## Effective resolution and d_50

In [ ]:
bts["effective_resolution"] = bts["radius"] / bts["n_sectors"]

print("Effective resolution (m):")
print(bts["effective_resolution"].describe().to_string())

bts["d_50"] = bts["effective_resolution"] * np.log(2)

print("\n50% detection threshold d_50 (m):")
print(bts["d_50"].describe().to_string())

## Detection probability function

In [ ]:
def detection_probability(trip_distance, radius, n_sectors):
    r_eff = radius / n_sectors
    if r_eff <= 0:
        return np.ones_like(trip_distance, dtype=float)
    return 1 - np.exp(-np.asarray(trip_distance) / r_eff)

## Figure -- Detection curves (Fig. 3 in paper)

In [ ]:
dx = np.linspace(0, 10_000, 500)

q_urban = bts["effective_resolution"].quantile(0.05)
q_suburban = bts["effective_resolution"].quantile(0.50)
q_rural = bts["effective_resolution"].quantile(0.95)

urban = bts.iloc[(bts["effective_resolution"] - q_urban).abs().argmin()]
suburban = bts.iloc[(bts["effective_resolution"] - q_suburban).abs().argmin()]
rural = bts.iloc[(bts["effective_resolution"] - q_rural).abs().argmin()]

fig, ax = plt.subplots(figsize=(7, 4.5))
for site, label, color in [
    (urban, "Urban (P5)", "#e41a1c"),
    (suburban, "Suburban (P50)", "#377eb8"),
    (rural, "Rural (P95)", "#4daf4a"),
]:
    delta = detection_probability(dx, site["radius"], site["n_sectors"])
    r_eff = site["radius"] / site["n_sectors"]
    ax.plot(dx, delta, color=color, lw=2,
            label=f'{label}: $r_{{eff}}$={r_eff:.0f} m '
                  f'(r={site["radius"]:.0f}, $n_s$={site["n_sectors"]:.0f})')

ax.axhline(0.5, color="grey", ls=":", lw=0.8)
ax.set_xlabel("Trip distance (m)")
ax.set_ylabel("Detection probability $\\delta(\\Delta x)$")
ax.set_title("Detection probability by site type")
ax.legend(fontsize=8)
ax.set_xlim(0, 10_000)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(FIGURES / "fig_detection_curves.pdf", bbox_inches="tight")
plt.show()

## Figure -- d_50 histogram (Fig. 4 in paper)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(bts["d_50"], bins=60, edgecolor="white", alpha=0.8, color="#377eb8")
ax.axvline(bts["d_50"].median(), color="red", ls="--", lw=1,
           label=f'Median = {bts["d_50"].median():.0f} m')
ax.axvline(bts["d_50"].quantile(0.95), color="orange", ls="--", lw=1,
           label=f'P95 = {bts["d_50"].quantile(0.95):.0f} m')
ax.set_xlabel("$d_{50}$ (m)")
ax.set_ylabel("Number of sites")
ax.set_title("Distribution of 50% detection threshold")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / "fig_d50_histogram.pdf", bbox_inches="tight")
plt.show()

## Monte Carlo validation

In [ ]:
def simulate_detection(radius, n_sectors, trip_distances, n_sim=10_000, seed=42):
    rng = np.random.default_rng(seed)
    trip_distances = np.asarray(trip_distances)
    results = np.zeros(len(trip_distances))

    d0 = radius * np.sqrt(rng.uniform(0, 1, n_sim))
    theta0 = rng.uniform(0, 2 * np.pi, n_sim)
    x0 = d0 * np.cos(theta0)
    y0 = d0 * np.sin(theta0)

    sector0 = np.floor(theta0 * n_sectors / (2 * np.pi)).astype(int) % n_sectors

    for i, dx in enumerate(trip_distances):
        phi = rng.uniform(0, 2 * np.pi, n_sim)
        x1 = x0 + dx * np.cos(phi)
        y1 = y0 + dx * np.sin(phi)

        d1 = np.hypot(x1, y1)
        theta1 = np.arctan2(y1, x1) % (2 * np.pi)
        sector1 = np.floor(theta1 * n_sectors / (2 * np.pi)).astype(int) % n_sectors

        detected = (d1 > radius) | (sector1 != sector0)
        results[i] = detected.mean()

    return results

## Figure -- Monte Carlo validation (Fig. 8 in paper)

In [ ]:
dx_mc = np.linspace(0, 5000, 40)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

for ax, (site, label) in zip(axes, [
    (urban, "Urban (P5)"),
    (suburban, "Suburban (P50)"),
    (rural, "Rural (P95)"),
]):
    r = site["radius"]
    ns = int(site["n_sectors"])
    r_eff = r / ns

    dx_fine = np.linspace(0, 5000, 200)
    delta_param = detection_probability(dx_fine, r, ns)

    delta_mc = simulate_detection(r, ns, dx_mc, n_sim=20_000)

    ax.plot(dx_fine, delta_param, "k-", lw=1.5, label="Parametric")
    ax.plot(dx_mc, delta_mc, "ro", ms=4, alpha=0.7, label="Monte Carlo")
    ax.axhline(0.5, color="grey", ls=":", lw=0.8)
    ax.axvline(r_eff * np.log(2), color="grey", ls=":", lw=0.8)
    ax.set_title(f"{label}\n$r$={r:.0f}m, $n_s$={ns}, $r_{{eff}}$={r_eff:.0f}m",
                 fontsize=9)
    ax.set_xlabel("Trip distance (m)")
    ax.set_xlim(0, 5000)
    ax.legend(fontsize=7)

axes[0].set_ylabel("Detection probability")
plt.suptitle("Parametric vs. Monte Carlo detection probability", y=1.02)
plt.tight_layout()
plt.savefig(FIGURES / "fig_mc_validation.pdf", bbox_inches="tight")
plt.show()

## Save outputs

In [ ]:
detection = bts[["bts_id", "n_sectors", "radius", "mean_height",
                 "effective_resolution", "d_50", "lat", "lon"]].copy()
detection.to_parquet(DATA / "detection_floor.parquet", index=False)
print(f"Saved: detection_floor.parquet ({len(detection):,} rows)")